In [1]:
import sys
sys.path.append('../src')

import torch

import selex_distribution, specialized_models, utils, tree, energy_models

/home/scrotti/Aptamer2025py/experiments/../src/sampling.py:2: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [21]:
from importlib import reload
reload(tree)
reload(selex_distribution)
reload(specialized_models)

<module 'specialized_models' from '/home/scrotti/Aptamer2025py/experiments/../src/specialized_models.py'>

In [25]:
L, q = 10, 3
n_rounds = 3

k = torch.randn(L, q)
h = torch.randn(L, q)

model = specialized_models.IndepSitesMultiRoundDistribution(k, h, n_rounds)

In [26]:
x = utils.random_data(4, L, q)

def dot(h, x):
    return (x * h).sum((-1,-2))

In [33]:
t = 1
model.tree.ancestors_of(t-1)

tensor([0])

In [36]:
for t in range(n_rounds):
    print(model.selection_energy_up_to_round(x, t) + t*dot(x, h))

tensor([0., 0., 0., 0.])
tensor([-4.7684e-07, -2.3842e-07,  2.3842e-07,  0.0000e+00],
       grad_fn=<AddBackward0>)
tensor([-9.5367e-07, -4.7684e-07,  4.7684e-07,  0.0000e+00],
       grad_fn=<AddBackward0>)


In [209]:
L, q = 10, 3
n_rounds = 2
n_modes = 1

tr = tree.Tree()
for t in range(n_rounds-1):
    tr.add_node(t-1)

n_selection_rounds = n_rounds - 1
selected_modes = torch.randint(2, (n_selection_rounds, n_modes)).to(torch.bool)
k = torch.randn(L, q)
hs = [torch.randn(L, q) for _ in range(n_modes)]
Ns0 = energy_models.IndepSites(k)
modes = [energy_models.IndepSites(h) for h in hs]
ps = selex_distribution.MultiModeDistribution(*modes)
model = specialized_models.IndepSitesMultiRoundDistribution(k, h, n_rounds)
chemical_potential = torch.rand(n_selection_rounds, n_modes)
selection_strength = torch.rand(n_selection_rounds)
model = selex_distribution.MultiRoundDistribution(
    Ns0, ps, tr, selected_modes=selected_modes,
    chemical_potential=chemical_potential,
    selection_strength=selection_strength
    )